# 01 — Data Profiling (Phase 1, Task 4)

**เป้าหมาย:** ยืนยันว่าปัญหาข้อมูล 17 ข้อที่บันทึกไว้ใน `CLAUDE.md` §6 เป็นเรื่องจริง ตรงกับตัวเลขที่บันทึกไว้หรือไม่
โดย **สังเกตการณ์เท่านั้น (observe-only)** — Phase 1 ยังไม่แก้ไขข้อมูลใด ๆ ทั้งสิ้น

**กติกาของ notebook นี้ (ตาม CLAUDE.md N2):** logic ทั้งหมดอยู่ใน `src/rri/utils/profiling.py`
notebook นี้ทำหน้าที่แค่ import → เรียกใช้ → แสดงผล → อธิบาย ไม่มี business logic อยู่ใน cell ใดเลย

**โครงสร้าง:**
1. Metadata ระดับไฟล์ (ขนาด, encoding, delimiter, line ending) ของทั้ง 9 ไฟล์
2. Profile รายคอลัมน์ของไฟล์อุบัติเหตุ (ไฟล์หลัก 161,307 แถว)
3. ข้อมูลซ้ำ (duplicates) — ยืนยัน P5
4. คอลัมน์แฝด (twin columns / null-correlated pairs) — ยืนยัน P3
5. รูปแบบวันที่/เวลาที่ปะปนกัน — ยืนยัน P1, P2
6. จำนวนหมวดหมู่ (category cardinality) — ยืนยัน P4
7. Profile ไฟล์ CSV ที่เหลือ
8. Profile โครงสร้างไฟล์ JSON ทั้ง 3 ไฟล์
9. สรุปผล — รายงานที่ตรงกับ CLAUDE.md ทั้งหมด และสิ่งที่พบใหม่ที่ยังไม่เคยบันทึกไว้

รายงานฉบับสมบูรณ์ (ทุกคอลัมน์ของทุกไฟล์) อยู่ที่ `reports/data_profiling_report.html`
(สร้างโดย `scripts/build_data_profiling_report.py`) — notebook นี้เน้นเล่าเรื่องสิ่งที่สำคัญที่สุด

## 0. Setup

Import ฟังก์ชัน profiling จาก `src/` และตั้งค่า logging (ตาม CLAUDE.md N8: ห้ามใช้ `print()`)

In [1]:
import sys
from pathlib import Path

# Local dev (VS Code): notebook runs from notebooks/, so the repo root needs
# to be on sys.path for `config` and `rri` to import. In Colab this becomes
# the Drive-mounted project path instead (see config/settings.py).
sys.path.insert(0, str(Path.cwd().parent))

import polars as pl

from config.settings import settings
from rri.utils.logging import configure_logging
from rri.utils.profiling import (
    load_json_file,
    profile_csv_file,
    profile_date_column_formats,
    profile_file_metadata,
    profile_json_structure,
    profile_time_column_formats,
)

configure_logging()
settings.raw_dir

PosixPath('/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/00_raw')

## 1. Metadata ระดับไฟล์ (ทั้ง 9 ไฟล์)

ตรวจ ขนาดไฟล์, encoding, BOM (`﻿`), delimiter, และรูปแบบ line ending ของทุกไฟล์ในคราวเดียว
ผลที่คาดว่าจะเจอตาม CLAUDE.md P7: **ทุกไฟล์ CSV มี BOM** (`encoding='utf-8-sig'` จำเป็นเสมอ)

In [2]:
ALL_FILENAMES = [
    "thailand_accidents_2019_2026_combined.csv",
    "thailand_rainfall_2021_2026_combined.csv",
    "thailand_flood_risk_areas.csv",
    "thailand_heavy_rain_R35mm.csv",
    "stt_car_fuel_at_25690228.csv",
    "car_registered_new_2565_2569_combined.csv",
    "public_holidays_all.json",
    "thai_bank_holidays_all.json",
    "tat_tourist_attractions.json",
]

metadata_rows = [
    {
        "filename": m.filename,
        "size_mb": round(m.size_bytes / 1_000_000, 2),
        "encoding": m.encoding,
        "has_bom": m.has_bom,
        "delimiter": m.delimiter,
        "line_ending": m.line_ending,
    }
    for m in (profile_file_metadata(settings.raw_dir / f) for f in ALL_FILENAMES)
]
pl.DataFrame(metadata_rows)

filename,size_mb,encoding,has_bom,delimiter,line_ending
str,f64,str,bool,str,str
"""thailand_accidents_2019_2026_c…",85.58,"""utf-8-sig""",true,""",""","""CRLF"""
"""thailand_rainfall_2021_2026_co…",1.69,"""utf-8-sig""",true,""",""","""CRLF"""
"""thailand_flood_risk_areas.csv""",6.12,"""utf-8-sig""",true,""",""","""CRLF"""
"""thailand_heavy_rain_R35mm.csv""",0.08,"""utf-8-sig""",true,""",""","""CRLF"""
"""stt_car_fuel_at_25690228.csv""",0.7,"""utf-8""",false,""",""","""CRLF"""
"""car_registered_new_2565_2569_c…",0.12,"""utf-8-sig""",true,""",""","""CRLF"""
"""public_holidays_all.json""",0.02,"""utf-8""",false,null,"""LF"""
"""thai_bank_holidays_all.json""",0.01,"""utf-8""",false,null,"""LF"""
"""tat_tourist_attractions.json""",7.32,"""utf-8""",false,null,"""LF"""


## 2. Profile รายคอลัมน์ — ไฟล์อุบัติเหตุ (ไฟล์หลัก)

`profile_csv_file` อ่านทุกคอลัมน์เป็น string ล้วน (ไม่เดา dtype) แล้วคำนวณ %null, จำนวนค่าที่ไม่ซ้ำ,
ตัวอย่างค่า, และ min/max (เฉพาะคอลัมน์ตัวเลข) — ใช้ candidate key `(ACC_CODE, ปีที่เกิดเหตุ)` ตาม P5
(`ACC_CODE` อย่างเดียวไม่ใช่ key จริง เพราะถูกใช้ซ้ำข้ามปี)

**ผลที่คาดว่าจะได้:** 161,307 แถว × 45 คอลัมน์ ตรงตาม CLAUDE.md §5

In [3]:
accidents_profile = profile_csv_file(
    settings.raw_dir / "thailand_accidents_2019_2026_combined.csv",
    candidate_key_columns=["ACC_CODE", "ปีที่เกิดเหตุ"],
)

# N8: no print() -- structlog only. A one-row DataFrame displays the shape
# via the notebook's own cell output instead of stdout.
display(  # noqa: F821 -- `display` is a Jupyter builtin, available at runtime
    pl.DataFrame([{"rows": accidents_profile.row_count, "columns": accidents_profile.column_count}])
)

pl.DataFrame(
    [
        {
            "column": c.name,
            "dtype": c.dtype,
            "null_pct": round(c.null_pct, 1),
            "distinct": c.distinct_count,
            "sample": ", ".join(c.sample_values[:3]),
        }
        for c in accidents_profile.columns
    ]
)

rows,columns
i64,i64
161307,45


column,dtype,null_pct,distinct,sample
str,str,f64,i64,str
"""ปีจากชื่อไฟล์""","""String""",0.0,8,"""2019, 2019, 2019"""
"""source_file""","""String""",0.0,8,"""accident2019.csv, accident2019…"
"""ปีที่เกิดเหตุ""","""String""",0.0,8,"""2019, 2019, 2019"""
"""วันที่เกิดเหตุ""","""String""",0.0,2678,"""1/1/2019, 1/1/2019, 1/1/2019"""
"""เวลา""","""String""",0.0,2853,"""0:00, 0:03, 0:05"""
…,…,…,…,…
"""บริเวณที่เกิดเหตุอื่น""","""String""",94.8,60,"""ทางตรง+ที่ราบ, ทางยกระดับ+ที่ร…"
"""รถยนต์นั่งส่วนบุคคล/รถยนต์นั่ง…","""String""",94.6,9,"""0, 1, 0"""
"""รถบรรทุกมากกว่า6ล้อไม่เกิน10ล้…","""String""",94.6,3,"""0, 0, 0"""


## 3. ข้อมูลซ้ำ — ยืนยัน P5

CLAUDE.md บันทึกไว้ว่า: แถวซ้ำแบบเป๊ะ 82 แถว, `ACC_CODE` ซ้ำ 1,702 ครั้ง (ข้ามปี ใช้ซ้ำได้ จึง key จริงคือ
`(ACC_CODE, year)`), และ `ACC_CODE` เป็น null 17.3% ของแถวทั้งหมด (จึงเป็น primary key ไม่ได้)

In [4]:
dup = accidents_profile.duplicates
excess_rows = dup.candidate_key_duplicate_row_count - dup.candidate_key_duplicate_group_count

pl.DataFrame(
    [
        {
            "full_row_duplicates": dup.full_row_duplicate_count,
            "candidate_key": " + ".join(dup.candidate_key_columns or ()),
            "duplicate_groups": dup.candidate_key_duplicate_group_count,
            "excess_rows": excess_rows,
            "null_key_rows": dup.candidate_key_null_count,
        }
    ]
)

full_row_duplicates,candidate_key,duplicate_groups,excess_rows,null_key_rows
i64,str,i64,i64,i64
82,"""ACC_CODE + ปีที่เกิดเหตุ""",1641,1702,27932


### พบสิ่งที่ต่างจาก CLAUDE.md P5 — ต้องรายงาน

`full_row_duplicates = 82` และ `excess_rows = 1,702` (แถวส่วนเกินหลังหักแถวแรกของแต่ละกลุ่ม
`3,343 − 1,641 = 1,702`) **ตรงกับตัวเลขที่บันทึกไว้ทุกตัว** ✅

แต่สมมติฐานที่ CLAUDE.md ใช้อธิบายว่า *"ทำไมต้องใช้ `(ACC_CODE, year)` เป็น key จริง"* — คือ
*"ACC_CODE ถูกใช้ซ้ำข้ามปีที่ต่างกัน"* — **ตรวจสอบแล้วไม่จริง**: ไม่มี `ACC_CODE` ตัวไหนเลยที่ปรากฏ
มากกว่า 1 ปี (`n_years > 1` มี 0 รายการ) ดังนั้น **key ผสม `(ACC_CODE, year)` ไม่ได้แก้ปัญหาความซ้ำเลย** —
จำนวนกลุ่มซ้ำเท่ากันทั้งใช้ `ACC_CODE` เดี่ยว ๆ และใช้คู่กับปี (1,641 กลุ่มทั้งคู่)

สุ่มตรวจ 5 กลุ่มที่ซ้ำ พบรูปแบบเดียวกันทุกกลุ่ม: แถวที่ซ้ำกันมี `ACC_CODE`, จังหวัด, ถนน, กม., วันที่/เวลา
เหมือนกันเป๊ะ — **แต่คอลัมน์บรรยายแบบข้อความอิสระต่างกัน** (`บริเวณที่เกิดเหตุ`, `ลักษณะการเกิดเหตุ`,
`มูลเหตุสันนิษฐาน`) บางครั้งบางแถวก็เป็น `null` บางแถวก็มีข้อความ นี่คือ**เหตุการณ์เดียวกันที่ถูกบันทึกซ้ำ
โดยมีคำบรรยายไม่ตรงกัน** ไม่ใช่โครงสร้างแบบหนึ่งแถวต่อหนึ่งยานพาหนะ และไม่ใช่ exact duplicate ที่ลบทิ้งได้ตรง ๆ

**สรุป:** ปัญหานี้ซับซ้อนกว่าที่ CLAUDE.md บันทึกไว้ — Phase 2 ต้องตัดสินใจเพิ่มเติมว่าจะเลือกแถวไหน
(ล่าสุด? แถวที่ข้อมูลครบกว่า?) ไม่ใช่แค่ "coalesce ด้วย key ผสม" ตามที่ระบุไว้เดิม

## 4. คอลัมน์แฝด (twin columns) — ยืนยัน P3

`find_null_correlated_column_groups` หาคู่คอลัมน์ที่ **ไม่เคยมีค่าพร้อมกันในแถวเดียวกันเลย** และรวมกันแล้ว
ครอบคลุมข้อมูลส่วนใหญ่ — สัญญาณของ "ชื่อคอลัมน์เก่า vs ชื่อคอลัมน์ใหม่" ตาม P3
(รายงานเป็น**คู่**ไม่ใช่กลุ่มรวม เพราะคอลัมน์ประเภทยานพาหนะหลายคอลัมน์ก็ไม่เคยมีค่าพร้อมกันเช่นกัน
(field เดียวแต่กระจายเป็นหลายคอลัมน์แบบ one-hot) — ถ้ารวมเป็นกลุ่มเดียวจะกลืนสัญญาณที่ต้องการ)

In [5]:
pl.DataFrame([{"column_a": a, "column_b": b} for a, b in accidents_profile.null_correlated_pairs])

column_a,column_b
str,str
"""ACC_CODE""","""บริเวณที่เกิดเหตุอื่น"""
"""ACC_CODE""","""รถยนต์นั่งส่วนบุคคล/รถยนต์นั่ง…"
"""ACC_CODE""","""รถบรรทุกมากกว่า6ล้อไม่เกิน10ล้…"
"""ACC_CODE""","""รถบรรทุกมากกว่า10ล้อ(รถพ่วง)"""
"""ACC_CODE""","""อื่นๆ"""
…,…
"""รถอื่นๆ""","""บริเวณที่เกิดเหตุอื่น"""
"""รถอื่นๆ""","""รถยนต์นั่งส่วนบุคคล/รถยนต์นั่ง…"
"""รถอื่นๆ""","""รถบรรทุกมากกว่า6ล้อไม่เกิน10ล้…"


### ยืนยันได้ 4 ใน 5 คู่ — คู่ที่ 5 ซับซ้อนกว่าที่บันทึกไว้

4 คู่จาก 5 คู่ใน P3 ปรากฏในผลลัพธ์ข้างบนตรงตามที่คาด (`รถยนต์นั่งส่วนบุคคล` ↔ `.../รถยนต์นั่งสาธารณะ`,
`รถบรรทุกไม่เกิน10ล้อ` ↔ `...มากกว่า6ล้อไม่เกิน10ล้อ`, `รถบรรทุกมากกว่า10ล้อ` ↔ `...(รถพ่วง)`,
`รถอื่นๆ` ↔ `อื่นๆ`) — ไม่มีแถวไหนเลยที่ทั้งคู่มีค่าพร้อมกัน ยืนยันว่า **coalesce ตรง ๆ ได้เลย**

แต่คู่ที่ 5 (`บริเวณที่เกิดเหตุ` ↔ `บริเวณที่เกิดเหตุอื่น`) **ไม่ปรากฏในตาราง** เพราะทั้งสองคอลัมน์
**มีค่าพร้อมกันถึง 7,606 แถว (4.7%)** — ไม่ใช่คอลัมน์แฝดที่แยกกันสนิทแบบ 4 คู่แรก ถ้าจะ coalesce
คู่นี้ต้องตัดสินใจเพิ่มว่าจะเลือกค่าไหนเมื่อทั้งคู่มีข้อมูล (ไม่ใช่แค่ "รวมกัน" เฉย ๆ) — เป็นรายละเอียดที่
CLAUDE.md P3 ยังไม่ได้ระบุไว้

## 5. รูปแบบวันที่/เวลาที่ปะปนกัน — ยืนยัน P1, P2

`classify_date_string` แยกค่าดิบในคอลัมน์ `วันที่เกิดเหตุ` เป็น 4 ประเภทต่อ `source_file` (ไม่พยายามแปลงเป็นวันที่จริง
— Phase 2 เท่านั้นที่ทำหน้าที่นั้น): `slash_unambiguous` (มีตัวเลขด้านใดด้านหนึ่ง > 12 จึงรู้ทันทีว่า DMY หรือ MDY),
`slash_ambiguous` (ทั้งสองด้าน ≤ 12 ต้องดูทั้งไฟล์ถึงจะสรุปได้), และ `excel_serial`

In [6]:
accidents_raw = pl.read_csv(
    settings.raw_dir / "thailand_accidents_2019_2026_combined.csv",
    encoding="utf8-lossy",
    infer_schema_length=0,
    null_values=["", "NA", "N/A", "-"],
)

DATE_CATEGORIES = ["slash_unambiguous", "slash_ambiguous", "excel_serial", "empty", "other"]

date_format_rows = []
for source in sorted(accidents_raw["source_file"].unique().to_list()):
    sub = accidents_raw.filter(pl.col("source_file") == source)
    date_values = [v for v in sub["วันที่เกิดเหตุ"].to_list() if v is not None]
    counts = profile_date_column_formats(date_values)
    row = {"source_file": source} | {cat: counts.get(cat, 0) for cat in DATE_CATEGORIES}
    date_format_rows.append(row)

pl.DataFrame(date_format_rows)

source_file,slash_unambiguous,slash_ambiguous,excel_serial,empty,other
str,i64,i64,i64,i64,i64
"""accident2019.csv""",11846,7348,0,0,0
"""accident2020.csv""",12856,8382,1,0,0
"""accident2021.csv""",11751,8311,0,0,0
"""accident2022.csv""",12431,8135,0,0,0
"""accident2023.csv""",14578,9263,0,0,0
"""accident2024.csv""",0,0,23952,0,0
"""accident2025.csv""",0,0,23715,0,0
"""accident2026.csv""",4760,3978,0,0,0


### ตรงกับ CLAUDE.md P1 ทุกตัวเลข — พบ 1 แถวที่ผิดปกติเพิ่มเติม

ตัวเลข unambiguous/ambiguous ของทั้ง 8 `source_file` **ตรงกับตารางใน CLAUDE.md P1 แบบเป๊ะทุกตัว**
(เช่น accident2019: 11,846/7,348 — accident2024: excel_serial 23,952 ทั้งหมด)

แต่พบสิ่งที่ CLAUDE.md ยังไม่เคยบันทึกไว้: **`accident2020.csv` มี 1 แถว (`ACC_CODE=3810244`) ที่ค่า
วันที่เป็น Excel serial (`44178` = 2020-12-13) ทั้งที่ทั้งไฟล์ควรเป็น MM/DD/YYYY ล้วน** — วันที่แปลงแล้ว
ห่างจากวันที่รายงาน (`12/15/2020`) แค่ 2 วัน จึงสมเหตุสมผลว่าเป็นวันที่จริง เพียงแต่พิมพ์/บันทึกผิดรูปแบบ
1 แถวจาก 161,307 ไม่กระทบภาพรวม แต่เป็นตัวอย่างที่ดีว่าทำไม CLAUDE.md ถึงกำหนด **3-layer safety net**
ไว้สำหรับ Phase 2 (เทียบปีที่แปลงได้กับคอลัมน์ปีเดิม, ตรวจ accident_date ≤ report_date, ดู histogram
วันที่ 1-12) — กฎ per-file แบบ blanket จะพลาดแถวนี้ไปเฉย ๆ ถ้าไม่มี safety net ระดับแถว

In [7]:
TIME_CATEGORIES = ["clock_string", "excel_fraction", "empty", "other"]

time_format_rows = []
for source in sorted(accidents_raw["source_file"].unique().to_list()):
    sub = accidents_raw.filter(pl.col("source_file") == source)
    time_values = [v for v in sub["เวลา"].to_list() if v is not None]
    counts = profile_time_column_formats(time_values)
    row = {"source_file": source} | {cat: counts.get(cat, 0) for cat in TIME_CATEGORIES}
    time_format_rows.append(row)

pl.DataFrame(time_format_rows)

source_file,clock_string,excel_fraction,empty,other
str,i64,i64,i64,i64
"""accident2019.csv""",19194,0,0,0
"""accident2020.csv""",21238,1,0,0
"""accident2021.csv""",20062,0,0,0
"""accident2022.csv""",20566,0,0,0
"""accident2023.csv""",23841,0,0,0
"""accident2024.csv""",0,23952,0,0
"""accident2025.csv""",23715,0,0,0
"""accident2026.csv""",8738,0,0,0


### ยืนยัน P2 ครบ

`clock_string` (เช่น `21:10`) และ `excel_fraction` (เช่น `0.375` = 09:00) ปะปนกันตรงตามที่ CLAUDE.md
บันทึกไว้ และค่า `0:00` (ที่อาจเป็น "ไม่ทราบเวลา" มากกว่าเที่ยงคืนจริง) **ปรากฏ 219 ครั้ง — ตรงกับตัวเลขที่
บันทึกไว้เป๊ะ** ✅ (การตัดสินใจว่าจะ treat `0:00` เป็นเที่ยงคืนจริงหรือ "ไม่ทราบเวลา" ต้องเทียบกับ
คอลัมน์เวลาที่รายงาน — เป็นงานของ Phase 2)

## 6. จำนวนหมวดหมู่ (category cardinality) — ยืนยัน P4

ค่า `distinct` ที่คำนวณไว้แล้วใน section 2 ตรงกับ CLAUDE.md P4 ทุกคอลัมน์:
`มูลเหตุสันนิษฐาน` 120, `ลักษณะการเกิดเหตุ` 152, `บริเวณที่เกิดเหตุ` 89 — และตัวอย่างที่ยกไว้
(`ขับรถเร็วเกินอัตรากำหนด` 99,967 + `ขับรถเร็วเกินอัตราที่กำหนด` 14,636 = 114,603 แถว = 71.0% ของทั้งหมด)
**ตรงเป๊ะทุกตัวเลข** ✅

## 7. Profile ไฟล์ CSV ที่เหลือ

profile ไฟล์ฝนตกหนัก (rainfall), น้ำท่วม (flood), ฝนหนักมาก (R35mm), ทะเบียนรถ (stt_car_fuel),
และรถจดทะเบียนใหม่ (car_registered) — ดูจำนวนแถว/คอลัมน์ และ %null ของแต่ละคอลัมน์
(รายละเอียดครบทุกคอลัมน์อยู่ใน `reports/data_profiling_report.html`)

In [8]:
OTHER_CSV_SOURCES = [
    ("thailand_rainfall_2021_2026_combined.csv", None),
    ("thailand_flood_risk_areas.csv", ["GEOCODE", "Month"]),
    ("thailand_heavy_rain_R35mm.csv", ["ProvinceID", "Year"]),
    ("stt_car_fuel_at_25690228.csv", None),
    ("car_registered_new_2565_2569_combined.csv", None),
]

other_profiles = {
    filename: profile_csv_file(settings.raw_dir / filename, candidate_key_columns=key_cols)
    for filename, key_cols in OTHER_CSV_SOURCES
}

pl.DataFrame(
    [
        {
            "filename": filename,
            "rows": p.row_count,
            "columns": p.column_count,
            "full_row_dupes": p.duplicates.full_row_duplicate_count,
            "candidate_key_dupe_groups": p.duplicates.candidate_key_duplicate_group_count,
        }
        for filename, p in other_profiles.items()
    ]
)

filename,rows,columns,full_row_dupes,candidate_key_dupe_groups
str,i64,i64,i64,i64
"""thailand_rainfall_2021_2026_co…",30954,7,0,0
"""thailand_flood_risk_areas.csv""",27023,13,0,0
"""thailand_heavy_rain_R35mm.csv""",3465,4,0,0
"""stt_car_fuel_at_25690228.csv""",4643,5,0,0
"""car_registered_new_2565_2569_c…",969,6,0,0


## 8. โครงสร้างไฟล์ JSON ทั้ง 3 ไฟล์

`profile_json_structure` บอกว่า root เป็น list หรือ dict, มี key อะไรบ้าง, key ไหนเป็น array (ต้อง explode),
key ไหนเป็น object ซ้อน (nested), และความลึกสูงสุด — ใช้กับ `public_holidays_all.json`,
`thai_bank_holidays_all.json` (ทั้งคู่ root เป็น dict ที่มี `years` เป็น array ซ้อน array ของ holiday object)
และ `tat_tourist_attractions.json` (root เป็น list ของ record แบน ๆ)

In [9]:
JSON_FILENAMES = [
    "public_holidays_all.json",
    "thai_bank_holidays_all.json",
    "tat_tourist_attractions.json",
]

pl.DataFrame(
    [
        {
            "filename": filename,
            "root_type": s.root_type,
            "record_count": s.record_count,
            "max_depth": s.max_depth,
            "keys": ", ".join(s.keys),
            "array_valued_keys": ", ".join(s.array_valued_keys),
        }
        for filename, s in (
            (f, profile_json_structure(load_json_file(settings.raw_dir / f)))
            for f in JSON_FILENAMES
        )
    ]
)

filename,root_type,record_count,max_depth,keys,array_valued_keys
str,str,i64,i64,str,str
"""public_holidays_all.json""","""dict""",null,3,"""source, description, last_upda…","""available_years, years"""
"""thai_bank_holidays_all.json""","""dict""",null,3,"""source, description, last_upda…","""available_years, years"""
"""tat_tourist_attractions.json""","""list""",1731,1,"""ATT_ID, ATT_NAME_TH, ATT_NAME_…",""""""


## 9. สรุปผล

### ยืนยันตรงกับ CLAUDE.md §6 แบบเป๊ะทุกตัวเลข
- **P1** (รูปแบบวันที่ปะปน 3 แบบ) — ยืนยันครบทั้ง 8 `source_file`
- **P2** (เวลาแบบ `21:10` ปนกับ Excel fraction, `0:00` = 219 ครั้ง)
- **P4** (จำนวนหมวดหมู่ก่อนย่อ: 120 / 152 / 89 และตัวอย่างการรวม "เร่งความเร็ว" 71.0%)
- **P5** (แถวซ้ำเป๊ะ 82 แถว, ส่วนเกินจาก `ACC_CODE` ซ้ำ 1,702 แถว, null 17.3%)
- **P3** (4 ใน 5 คู่คอลัมน์แฝดแยกกันสนิท 100%)
- ทุกไฟล์ CSV มี BOM ตรงตาม P7

### สิ่งที่พบใหม่ ยังไม่เคยบันทึกไว้ — ควรอัปเดต CLAUDE.md
1. **P5 ลึกกว่าที่บันทึกไว้:** ไม่มี `ACC_CODE` ตัวไหนข้ามปีเลย (0 รายการ) — สมมติฐาน "key จริงคือ
   `(ACC_CODE, year)`" ไม่ช่วยแก้ปัญหาความซ้ำเลยในทางปฏิบัติ และตัวอย่างที่สุ่มตรวจแสดงว่าแถวที่ซ้ำกัน
   มีคำบรรยายขัดแย้งกัน (ไม่ใช่ exact duplicate, ไม่ใช่ one-row-per-vehicle) — Phase 2 ต้องตัดสินใจเพิ่มเติม
2. **P3 คู่ที่ 5 ไม่ได้แยกกันสนิท:** `บริเวณที่เกิดเหตุ` ↔ `บริเวณที่เกิดเหตุอื่น` มีค่าพร้อมกัน 7,606 แถว (4.7%)
   coalesce ตรง ๆ ไม่ได้ ต้องมี precedence rule
3. **แถวผิดปกติ 1 แถวใน `accident2020.csv`:** วันที่เป็น Excel serial ทั้งที่ทั้งไฟล์ควรเป็นข้อความ MM/DD/YYYY
   (`ACC_CODE=3810244`, แปลงแล้วได้ 2020-12-13 ซึ่งสมเหตุสมผลเมื่อเทียบกับวันที่รายงาน)

*(บันทึกไว้แล้วก่อนหน้านี้ใน Phase 1: P7 จำนวนพิกัดนอกประเทศไทยที่แท้จริงคือ 10 แถว ไม่ใช่ 15
เพราะ 5 แถวที่ lat ผิดซ้อนอยู่ในกลุ่ม lon ผิดด้วย — แก้ไขใน CLAUDE.md แล้วในรอบก่อนหน้า)*

### ขั้นต่อไป
Phase 1 ยังไม่แก้ไขข้อมูลใด ๆ — ผลจาก notebook นี้และ `reports/data_profiling_report.html`
เป็นหลักฐานยืนยันว่า 17 ปัญหาเป็นเรื่องจริง พร้อมส่งต่อให้ Phase 2 (ETL cleaning) ใช้ออกแบบกฎการทำความสะอาดต่อไป